## File PATHS

In [1]:
QUESTION_JSON_PATH = 'questions/eval_questions.json'
RESULT_PATH = "results/naive_rag_results.csv"
RAW_SEC_FILE = 'raw_regulations/sec_reg.txt'

In [2]:
EMBED_MODEL = "text-embedding-3-large"
LLM_MODEL = "openai/gpt-oss-120b"

In [3]:
GITHUB_URL = "https://models.github.ai/inference" 

In [8]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_KEY = os.getenv("GROQ_KEY")
GITHUB_KEY = os.getenv("GITHUB_KEY")

# OPENROUTER_KEY = os.getenv("OPENROUTER_KEY")


In [5]:
import time
import os
import json
import pandas as pd
from tqdm import tqdm


from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document

from openai import OpenAI

In [11]:
os.environ["OPENAI_API_KEY"] = GITHUB_KEY
os.environ["OPENAI_API_BASE"] = GITHUB_URL

In [12]:
from groq import Groq
os.environ["GROQ_API_KEY"] = GROQ_KEY
client = Groq()

In [ ]:
# client = OpenAI(api_key=GITHUB_KEY, base_url=GITHUB_URL)

# Building Vector DB

In [13]:
import time
from tqdm import tqdm

def setup_naive_rag(file_path):
    print("Reading file and Chunking...")
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()

    # 1. Splitting Text
    text_splitter = RecursiveCharacterTextSplitter(
        
        chunk_size=2000,
        chunk_overlap=200,
        separators=["\n\n", "\n", " ", ""]
    )
    chunks = text_splitter.split_text(text)
    print(f"✂️ Cutting text to {len(chunks)} Chunks")

    # 2. Create Document
    docs = [Document(page_content=chunk) for chunk in chunks]
    
    # Config chunk_size for OpenAIEmbeddings
    embeddings = OpenAIEmbeddings(
        model=EMBED_MODEL,
        chunk_size=20 
    )
    
    print("Buiding Vector Database (FAISS) in Batch...")
    
    # 3. Building DB in first Batch
    batch_size = 20
    vectorstore = FAISS.from_documents(docs[:batch_size], embeddings)
    
    # 4. looping the rest Batch
    for i in tqdm(range(batch_size, len(docs), batch_size), desc="Embedding Progress"):
        batch_docs = docs[i : i + batch_size]
        
        # add to FAISS
        vectorstore.add_documents(batch_docs)
        
        # Avoid GitHub API (Rate Limit)
        time.sleep(1) 
        
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    
    print("\n✅ Build Naive RAG Retriever succeeded!")
    return retriever


กำลังอ่านไฟล์และทำ Chunking...
✂️ Cutting text to 32 Chunks
Buiding Vector Database (FAISS) in Batch...


Embedding Progress: 100%|████████████████████████████████████████████████████████████████| 1/1 [00:03<00:00,  3.21s/it]


✅ Build Naive RAG Retriever succeeded!


In [ ]:
retriever = setup_naive_rag(RAW_SEC_FILE)

In [14]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001B2FFC14500>, search_kwargs={'k': 5})

# Answering Questions

In [15]:
def generate_rag_answer(query, retriever):
    # 1. Searching data (Retrieve)
    retrieved_docs = retriever.invoke(query)
    # Create Context
    context = "\n---\n".join([doc.page_content for doc in retrieved_docs])
    

    prompt = f"""
    จงตอบคำถามต่อไปนี้โดยอิงจาก "ข้อมูลอ้างอิง" ที่กำหนดให้เท่านั้น 
    หากข้อมูลอ้างอิงไม่มีคำตอบที่ชัดเจน ให้ตอบว่า "ไม่พบข้อมูลในเอกสารอ้างอิง" ห้ามคิดคำตอบขึ้นมาเองเด็ดขาด
    
    ข้อมูลอ้างอิง:
    {context}
    
    คำถาม: {query}
    
    จงตอบกลับมาเป็นรูปแบบ JSON เท่านั้น โดยใช้โครงสร้างดังนี้:
    {{
        "legal_clause": ["ข้อ 13 (1)", "ข้อ 54 (2)"], // 📌 ระบุเฉพาะข้อกฎหมายที่ใช้เป็นฐานของคำตอบจริงๆ เท่านั้น
        "summary_answer": "สรุปคำตอบอย่างกระชับและตรงประเด็น"
    }}
    """


    
    try:
        # Prompt to LLM with response_format as json_object
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            temperature=0.0 # No creative
        )
        
        answer_data = json.loads(response.choices[0].message.content)
        
    except Exception as e:
        answer_data = {
            "legal_clause": "Error",
            "summary_answer": str(e)
        }
    
    return {
        "context": context,
        "legal_clause": answer_data.get("legal_clause", "ไม่ระบุ"),
        "summary_answer": answer_data.get("summary_answer", "ไม่พบคำตอบ")
    }

In [16]:
with open(QUESTION_JSON_PATH, 'r', encoding='utf-8') as f:
    eval_questions = json.load(f)

In [73]:

results = []
print("🚀 Start Answering with Naive RAG by question...")

for q in tqdm(eval_questions, desc="Testing RAG"):
    start_time = time.time()
    query = q['query']
    expected = q['expected_answer']
    
    rag_output = generate_rag_answer(query, retriever)
    print(rag_output)

    latency = time.time() - start_time
    
    results.append({
        "id": q.get('id', 'N/A'),
        "difficulty": q.get('difficulty', 'N/A'),
        "query": query,
        "expected_answer": expected,
        "Latency (s)": round(latency, 2),
        "expected_clause": ", ".join(q.get('reference_clause', [])),
        "retrieved_context": rag_output['context'],
        "naive_rag_clause": rag_output['legal_clause'], 
        "naive_rag_answer": rag_output['summary_answer'] 
    })


df_results = pd.DataFrame(results)

columns_order = [
    'id', 'difficulty', 'query', "Latency (s)",
    'expected_clause', 'naive_rag_clause', 
    'expected_answer', 'naive_rag_answer', 
    'retrieved_context'
]
df_results = df_results[columns_order]

df_results['naive_rag_clause'] = df_results['naive_rag_clause'].apply(lambda n: ', '.join(n))

df_results.to_csv(RESULT_PATH, index=False, encoding='utf-8-sig')

🚀 เริ่มทำการทดสอบ Naive RAG ทีละข้อ...


Testing RAG:  10%|███████                                                               | 1/10 [00:03<00:32,  3.62s/it]

{'context': 'กรณี                                                   ข้อความหน้าตั๋วเงิน\n\n1. ตั๋วเงินทุกกรณี                                    “ตั๋วเงินนี้เป็นหลักทรัพย์ และได้รับอนุญาตให้เสนอขายในวงจำกัด”\n\n2. ตั๋วเงิน PP10                                     “เปลี่ยนมือไม่ได้” หรือคำอื่นที่มีความหมายในทำนองเดียวกัน\n\n3. ตั๋วเงินที่ออกโดย FI*                            “ตั๋วเงินนี้ไม่ได้รับความคุ้มครองจากสถาบันคุ้มครองเงินฝาก”\n\n4. ตั๋วเงินที่เสนอขาย II**                          “มีวัตถุประสงค์ให้เปลี่ยนมือได้เฉพาะในกลุ่มผู้ลงทุนสถาบัน”\n\n                                                         หรือ คำอื่นที่มความหมายในทำนองเดียวกัน\n\n*FI หมายถึง ธนาคารพาณิชย์ บริษัทเงินทุน หรือบริษัทเครดิตฟองซิเอร์ตามกฎหมายว่าด้วยธุรกิจสถาบันการเงิน\n\n*II หมายถึง ผู้ลงทุนสถาบัน\n\n \n\nข้อ 36 การเสนอขายตั๋วเงินในลักษณะ PP10 หากผู้ได้รับอนุญาตจัดให้มีหลักประกันในการชำระหนี้ตามตั๋วเงินดังกล่าว ผู้ได้รับอนุญาตต้องดำเนินการให้ข้อตกลงเกี่ยวกับการวางหลักประกันดังกล่าวมีผลบังคับได้ตามกฎหมาย โดยคำนึ

Testing RAG:  20%|██████████████                                                        | 2/10 [00:08<00:36,  4.59s/it]

{'context': '“ตลาดหลักทรัพย์” หมายความว่า ตลาดหลักทรัพย์แห่งประเทศไทย\n\n“ผู้บริหาร” หมายความว่า\n\n(1) ผู้ที่ดำรงตำแหน่งบริหารสูงสุดของกิจการ\n\n(2) ผู้ที่ดำรงตำแหน่งระดับบริหาร 4 รายแรกนับต่อจากผู้ที่ดำรงตำแหน่งบริหารสูงสุดของกิจการลงมา และผู้ซึ่งดำรงตำแหน่งเทียบเท่ากับผู้ดำรงตำแหน่งระดับบริหารรายที่ 4 ทุกราย\n\n(3) ผู้ดำรงตำแหน่งระดับบริหารในสายงานบัญชีหรือสายงานการเงินที่เป็นระดับผู้จัดการฝ่ายขึ้นไปหรือเทียบเท่า\n\n“ที่ปรึกษาทางการเงิน” หมายความว่า ที่ปรึกษาทางการเงินที่ได้รับความเห็นชอบจากสำนักงาน\n\nภาค 1\n\nบททั่วไป\n\n________________\n\nหมวด 1\n\nขอบเขตของประกาศและหลักเกณฑ์ทั่วไป\n\n________________\n\nข้อ 3 ประกาศนี้ให้ใช้บังคับกับการขออนุญาตและการอนุญาตให้เสนอขายตราสารหนี้ที่ออกใหม่ต่อบุคคลในวงจำกัด และการเสนอขายหุ้นกู้แปลงสภาพต่อบุคคลที่มีลักษณะเฉพาะตามข้อ 4\nโดยกิจการดังต่อไปนี้\n\n(1) กิจการไทย ได้แก่\n\n     (ก) บริษัทจำกัดหรือบริษัทมหาชนจำกัดที่จัดตั้งขึ้นตามกฎหมายไทย\n\n     (ข) สาขาธนาคารพาณิชย์ต่างประเทศที่สามารถประกอบธุรกิจธนาคารพาณิชย์ได้ตามกฎหมายว่าด้วยธุรกิจสถาบั

...

{'context': 'เมื่อพ้นระยะเวลาที่สำนักงานกำหนดตามวรรคหนึ่ง หรือเมื่อผู้ขออนุญาตได้ปฏิบัติตามเงื่อนไขที่สำนักงานกำหนดแล้ว มิให้สำนักงานนำข้อเท็จจริงอันเป็นเหตุแห่งการสั่งการตามวรรคหนึ่งมาประกอบการพิจารณาคำขอในครั้งใหม่อีก\n\nในกรณีที่สำนักงานเห็นว่าเหตุที่ทำให้ผู้ขออนุญาตมีลักษณะไม่เป็นไปตามข้อ 56 เป็นเรื่องไม่ร้ายแรง หรือได้มีการแก้ไขหรือกำหนดมาตรการป้องกันแล้ว สำนักงานอาจไม่นำข้อเท็จจริงเกี่ยวกับการขาดลักษณะดังกล่าวมาใช้ประกอบการพิจารณาก็ได้\n\nส่วนที่ 2\n\nการสั่งการในกรณีที่ข้อเท็จจริงและพฤติการณ์\n\nเปลี่ยนแปลงไปภายหลังการอนุญาต\n\n________________\n\nข้อ 13 ในกรณีที่ปรากฏต่อสำนักงานภายหลังการอนุญาตให้เสนอขายตราสารหนี้ว่าข้อเท็จจริงและพฤติการณ์ที่สำนักงานนำมาพิจารณาเพื่ออนุญาตให้เสนอขายตราสารหนี้เปลี่ยนแปลงไปซึ่งหากข้อเท็จจริงและพฤติการณ์ที่เปลี่ยนแปลงไปนั้นปรากฏต่อสำนักงานก่อนการอนุญาต สำนักงานอาจไม่อนุญาตให้เสนอขายตราสารหนี้นั้น ให้สำนักงานมีอำนาจดังต่อไปนี้\n\n(1) สั่งให้ผู้ได้รับอนุญาต กรรมการ หรือผู้บริหารของผู้ได้รับอนุญาต ชี้แจงหรือเปิดเผยข้อมูลเพิ่มเติมภายในระยะเวลาที่กำหนด 

Testing RAG:  90%|███████████████████████████████████████████████████████████████       | 9/10 [04:19<00:33, 33.97s/it]

{'context': '(1) ไม่ใช่หุ้นกู้ที่มีอนุพันธ์แฝงตามส่วนที่ 6 ของภาคผนวก 2 ท้ายประกาศนี้\n\n(2) เป็นไปตามเงื่อนไขครบถ้วนดังนี้\n\n     (ก) กำหนดมูลค่าผลตอบแทนก่อนการใช้สิทธิในการแปลงสภาพไว้อย่างแน่นอนหรือเป็นอัตราที่ผันแปรตามอัตราดอกเบี้ยของสถาบันการเงินหรืออัตราดอกเบี้ยอื่น และไม่มีการกำหนดเงื่อนไขการจ่ายผลตอบแทนที่อ้างอิงกับปัจจัยอ้างอิงอื่นเพิ่มเติม\n\n(ข) มีข้อกำหนดในการแปลงสภาพที่ระบุให้แปลงสภาพเป็นหุ้นของผู้ออกหุ้นกู้เท่านั้น\n\n(ค) กำหนดราคาแปลงสภาพโดยไม่ผูกอยู่กับปัจจัยอ้างอิงอื่น\n\nข้อ 78 การเสนอขายหุ้นกู้แปลงสภาพที่ออกใหม่และหุ้นรองรับต่อผู้ถือหุ้นของบริษัทโดยบริษัทจดทะเบียนหรือบริษัทมหาชนจำกัดที่มีหน้าที่ตามมาตรา 56 ไม่ว่าหน้าที่นั้นจะเกิดจากการเสนอขายโดยบริษัทเองหรือโดยผู้ถือหุ้นของบริษัท ให้ถือว่าได้รับอนุญาตจากสำนักงานเมื่อเข้าลักษณะใดลักษณะหนึ่งดังต่อไปนี้\n\n(1) เป็นการเสนอขายตามมติที่ประชุมผู้ถือหุ้นที่ให้เสนอขายต่อผู้ถือหุ้นทุกรายอย่างเท่าเทียมกันตามสัดส่วนการถือหุ้น แต่จะไม่เสนอขายให้ผู้ถือหุ้นที่จะทำให้บริษัทมีหน้าที่ตามกฎหมายต่างประเทศ ซึ่งผู้ที่จะใช้สิทธิแปลงสภาพต้อ

Testing RAG: 100%|█████████████████████████████████████████████████████████████████████| 10/10 [04:59<00:00, 29.97s/it]

{'context': '(ก) เคยมีประวัติฝ่าฝืนหลักเกณฑ์หรือเงื่อนไขเกี่ยวกับการเสนอขายหลักทรัพย์ในเรื่องที่มีนัยสำคัญ\n\n     (ข) เคยถูกสำนักงานปฏิเสธคำขออนุญาตเสนอขายหุ้นที่ออกใหม่ เนื่องจากมีเหตุ\nที่มีนัยสำคัญอันควรสงสัยเกี่ยวกับกลไกการบริหารจัดการในลักษณะอย่างใดอย่างหนึ่งดังนี้\n\n           1. อาจไม่สามารถปฏิบัติต่อผู้ถือหุ้นได้อย่างเป็นธรรมโดยอาจมีการเอื้อประโยชน์ให้แก่ผู้ถือหุ้นกลุ่มใดกลุ่มหนึ่งซึ่งจะทำให้ได้เปรียบผู้ถือหุ้นรายอื่นหรือได้ประโยชน์มากกว่าผู้ถือหุ้นรายอื่นโดยไม่สมควร\n\n            2. อาจไม่สามารถรักษาสิทธิของผู้ถือหุ้นได้โดยทำให้บุคคลใดบุคคลหนึ่งได้รับประโยชน์ทางการเงินนอกเหนือจากที่พึงได้ตามปกติ หรือโดยทำให้บริษัทเสียประโยชน์ที่พึงได้รับ\n\n     (ค) เคยถูกสำนักงานปฏิเสธคำขออนุญาตหรือสั่งเพิกถอนการอนุญาตเสนอขายหลักทรัพย์ที่ออกใหม่ เนื่องจากมีเหตุที่ทำให้สงสัยเกี่ยวกับการเปิดเผยข้อมูลต่อประชาชนหรือต่อสำนักงานที่ไม่ครบถ้วน ไม่ตรงตามความเป็นจริง ไม่เพียงพอต่อการตัดสินใจลงทุน หรือทำให้ผู้ลงทุนสำคัญผิดซึ่งมีลักษณะเป็นการปกปิดหรืออำพราง หรือสร้างข้อมูลที่อาจไม่มีอยู่จริงในรายการหร

In [74]:
df_results

,id,difficulty,query,Latency (s),expected_clause,naive_rag_clause,expected_answer,naive_rag_answer,retrieved_context
0,Q_Vocab_Easy_01,ง่าย,คำว่า 'หุ้นกู้ด้อยสิทธิ' มีความหมายทางกฎหมายว่...,3.62,ข้อ 2,"ข้อ 34 (1), ข้อ 34 (2)",หมายถึง หุ้นกู้ที่ไม่มีประกัน ซึ่งมีการกำหนดเง...,หุ้นกู้ด้อยสิทธิหมายถึงหุ้นกู้ที่ให้สิทธิการรั...,กรณี ...
1,Q_Vocab_Easy_02,ง่าย,นิยามของคำว่า 'ผู้บริหาร' ครอบคลุมถึงตำแหน่ง '...,5.27,ข้อ 2,ข้อ 2,ครอบคลุมด้วย เนื่องจากบทนิยามระบุไว้อย่างชัดเจ...,"ใช่, คำจำกัดความของ 'ผู้บริหาร' รวมถึงผู้ดำรงต...",“ตลาดหลักทรัพย์” หมายความว่า ตลาดหลักทรัพย์แห่...
2,Q_Clause_Easy_03,ง่าย,เมื่อบริษัทได้รับอนุญาตให้เสนอขายตราสารหนี้แล้...,38.46,ข้อ 30 วรรคหนึ่ง,ข้อ 30,ต้องดำเนินการขายให้แล้วเสร็จภายใน 6 เดือนนับแต...,ต้องดำเนินการขายให้เสร็จภายใน 6 เดือน,(3) วัตถุประสงค์การใช้เงิน\n\n(4) แหล่งเงินทุน...
3,Q_Subclause_Medium_01,กลาง,สำหรับการเสนอขายตั๋วเงินแบบเฉพาะเจาะจง (PP10) ...,33.16,ข้อ 15 (1) (2),ข้อ 15,ตั๋วเงินที่เสนอขายต้องมีจำนวนไม่เกิน 10 ฉบับ ณ...,จำนวนฉบับของตั๋วเงินแบบ PP10 ที่เสนอขายให้ผู้ล...,ภาค 2\n\nลักษณะการเสนอขายตราสารหนี้ในวงจำกัด\n...
4,Q_Section_Medium_02,กลาง,สัญญาแต่งตั้งผู้แทนผู้ถือหุ้นกู้จะต้องระบุเรื่...,34.96,ข้อ 76 (4),ข้อ 76 (4),ต้องระบุอัตราและวิธีการจ่ายค่าตอบแทนไว้เป็นจำน...,สัญญาต้องระบุอัตราและวิธีการจ่ายค่าตอบแทนหรือบ...,ข้อ 24 ในกรณีที่เป็นการเสนอขายหุ้นกู้มีประกันห...
5,Q_Table_Hard_01,ยาก,ธนาคารพาณิชย์ที่ออกตั๋วเงินเพื่อเสนอขายในวงจำก...,35.75,"ข้อ 35, ข้อ 72","ข้อ 35 (3), ข้อ 72",ต้องพิมพ์ข้อความว่า 'ตั๋วเงินนี้ไม่ได้รับความค...,ต้องพิมพ์ข้อความว่า “ตั๋วเงินนี้ไม่ได้รับความค...,: หุ้นกู้ / พันธบัตร\n\n______________________...
6,Q_CrossCategory_Hard_02,ยาก,ถ้านิติบุคคลต่างประเทศต้องการเข้ามาเสนอขายตราส...,38.84,"ข้อ 23 (3), ข้อ 59 (1)",ข้อ 59 (1),กระทรวงการคลัง (ต้องแสดงได้ว่าดำเนินการภายใต้ข...,ผู้ขออนุญาตเป็นนิติบุคคลต่างประเทศต้องยื่นคำขอ...,: หุ้นกู้ / พันธบัตร\n\n______________________...
7,Q_CrossArticle_Hard_03,ยาก,หาก ก.ล.ต. พบว่าข้อเท็จจริงของบริษัทเปลี่ยนไปอ...,35.70,"ข้อ 13 (1), ข้อ 2",ข้อ 13 (1),สั่งได้ เนื่องจากข้อ 13 (1) ให้อำนาจสำนักงานสั...,"ใช่, ตามข้อ 13 (1) สำนักงานสามารถสั่งให้ผู้บริ...",เมื่อพ้นระยะเวลาที่สำนักงานกำหนดตามวรรคหนึ่ง ห...
8,Q_CrossArticle_Hard_04,ยาก,หากบริษัทต้องการเสนอขายพันธบัตรให้แก่กลุ่มเจ้า...,33.24,"ข้อ 30 วรรคสาม, ข้อ 15","ข้อ 30, ข้อ 30 (ยกเว้น)",ไม่ถูกบังคับ เนื่องจากข้อ 30 วรรคสาม กำหนดข้อย...,ไม่บังคับให้ขายพันธบัตรให้เสร็จภายใน 6 เดือน เ...,(1) ไม่ใช่หุ้นกู้ที่มีอนุพันธ์แฝงตามส่วนที่ 6 ...
9,Q_Exception_Hard_05,ยาก,บริษัทมหาชนที่จดทะเบียนในตลาดหลักทรัพย์แห่งหนึ...,40.67,"ข้อ 53 (2), ข้อ 54 (2)",ข้อ (2),ไม่ขาดคุณสมบัติ แม้ข้อ 53 (2) จะกำหนดลักษณะต้อ...,บริษัทที่เคยถูกศาลพิพากษาในคดีฉ้อโกงภายใน 10 ป...,(ก) เคยมีประวัติฝ่าฝืนหลักเกณฑ์หรือเงื่อนไขเกี...
